In [1]:
import os, sys

os.chdir(os.path.expanduser("~/QIAO0042/models/acv/facemask/"))
sys.path.insert(0, os.getcwd())
print("CWD:", os.getcwd())

CWD: /scratch-share/QIAO0042/models/acv/facemask


In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
import wandb

from unet import UNetV2
from losses import FocalDiceLoss
from metrics import confusion_matrix, f1_macro_from_cm, fbeta_present_gt_from_cm
from palette import NUM_CLASSES
from dataset import FaceParsingDataset
from split_utils import list_images, make_split, save_split
from augment import make_face_aug

In [3]:
aug_fn = make_face_aug(p_flip=0.5, p_geom=0.7, p_color=0.7, p_blur=0.15)
# Pipeline: flip+label_swap → ShiftScaleRotate → ColorJitter → optional GaussianBlur

/home/msai/qiao0042/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/scratch-share/QIAO0042/models/acv/facemask/augment.py:166: UserWarning: Argument(s) 'value, mask_value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(


In [4]:
import shutil, os, time as _t
from pathlib import Path

def copy_to_tmp(src_dirs: list[str], tmp_root: str = "/tmp/facemask") -> dict[str, str]:
    """
    Copy dataset directories to /tmp (local SSD on HPC nodes) for fast I/O.
    Returns a mapping {original_dir: tmp_dir}.

    On NFS/Lustre, reading 900 PNG files per epoch over the network adds
    significant overhead. /tmp is always on a local device.
    """
    mapping = {}
    for src in src_dirs:
        src_path = Path(src).resolve()
        dst_path = Path(tmp_root) / src_path.name
        if dst_path.exists():
            print(f"  /tmp cache already exists: {dst_path}")
        else:
            t0 = _t.time()
            shutil.copytree(src_path, dst_path)
            print(f"  copied {src_path} → {dst_path}  ({_t.time()-t0:.1f}s)")
        mapping[str(src)] = str(dst_path)
    return mapping

# Copy train images + masks to /tmp. Skip if already there.
# Set use_tmp=False if you are already on a local filesystem.
use_tmp = True
if use_tmp:
    tmp_map = copy_to_tmp(["train/images", "train/masks"])
    _img_dir  = tmp_map["train/images"]
    _mask_dir = tmp_map["train/masks"]
    print("Data dirs:", _img_dir, _mask_dir)
else:
    _img_dir  = "train/images"
    _mask_dir = "train/masks"

  /tmp cache already exists: /tmp/facemask/images
  /tmp cache already exists: /tmp/facemask/masks
Data dirs: /tmp/facemask/images /tmp/facemask/masks


In [ ]:
import numpy as np
import logging
from pathlib import Path
from PIL import Image as PILImage
from palette import rgb_to_label

logging.getLogger("torch._inductor").setLevel(logging.WARNING)
logging.getLogger("torch._dynamo").setLevel(logging.WARNING)

img_dir  = _img_dir
mask_dir = _mask_dir

seed      = 42
val_ratio = 0.1

batch_size        = 64
# Warmup must be calibrated in gradient steps, not epochs.
# Original: 5ep × 57steps = 285 warmup steps.
# bs=64 gives ~14 steps/epoch → need 285/14 ≈ 20 warmup epochs.
# With warmup_epochs=10 the lr hit 7e-3 after only 56 gradient steps (vs 228
# in the original), causing divergence before the model could stabilise.
lr                = 2e-2
warmup_epochs     = 20
epochs            = 300
base_width        = 23
dropout           = 0.3
aux_weight        = 0.4
focal_gamma       = 2.0
dice_w            = 0.85
label_smoothing   = 0.0
ohem_ratio        = 0.7
ema_decay         = 0.999
ckpt_path         = "checkpoints/v2_best.pt"
device            = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hard classes that score lowest — hat(14), ear_r(15), neck_l(16)
HARD_CLASSES      = {14, 15, 16}
hard_class_weight = 3.0   # sampling multiplier for images containing any hard class

use_bf16  = device.type == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"AMP dtype : {amp_dtype}")

torch.backends.cudnn.benchmark = True

# --- split ---
all_files = list_images(img_dir)
train_files, val_files = make_split(all_files, val_ratio=val_ratio, seed=seed)
save_split(train_files, val_files, out_dir="splits", tag=f"seed{seed}_vr{val_ratio}")
print(f"Split     : {len(train_files)} train / {len(val_files)} val")

# --- class weights (sqrt median-frequency balancing) ---
print("Computing class pixel frequencies ...")
pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
mask_lookup  = {
    p.stem: p for p in Path(mask_dir).iterdir()
    if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
}
for fn in train_files:
    stem     = Path(fn).stem
    mask_rgb = np.array(PILImage.open(mask_lookup[stem]).convert("RGB"), dtype=np.uint8)
    labels   = rgb_to_label(mask_rgb)
    for c in range(NUM_CLASSES):
        pixel_counts[c] += int((labels == c).sum())

freq             = pixel_counts / pixel_counts.sum()
median_freq      = float(np.median(freq[freq > 0]))
class_weights_np = np.where(freq > 0, np.sqrt(median_freq / freq), 1.0)
class_weights    = torch.tensor(class_weights_np, dtype=torch.float32)

print(f"{'cls':>4}  {'freq':>8}  {'weight':>8}")
for i, (f, w) in enumerate(zip(freq, class_weights_np)):
    print(f"  {i:2d}   {f:.5f}   {w:.3f}")

# --- datasets ---
train_ds = FaceParsingDataset(img_dir, mask_dir, file_list=train_files,
                              augment=aug_fn, cache=True)
val_ds   = FaceParsingDataset(img_dir, mask_dir, file_list=val_files,
                              augment=None, cache=True)

# --- weighted sampler: up-sample images containing hat/ear_r/neck_l ---
_sample_weights = torch.tensor([
    hard_class_weight if any(np.any(lbl == c) for c in HARD_CLASSES) else 1.0
    for lbl in train_ds._mask_cache
], dtype=torch.float)
sampler = WeightedRandomSampler(_sample_weights, num_samples=len(_sample_weights), replacement=True)
n_hard  = int((_sample_weights > 1).sum())
print(f"Hard-class images  : {n_hard} / {len(_sample_weights)}  (sampled at {hard_class_weight}×)")

train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                          num_workers=8, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=32,         shuffle=False,
                          num_workers=4, pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

# --- model ---
model = UNetV2(num_classes=NUM_CLASSES, base=base_width,
               dropout=dropout, deep_supervision=True).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# --- multi-GPU: DataParallel BEFORE torch.compile ---
num_gpus = torch.cuda.device_count()
if num_gpus > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel       : {num_gpus} GPUs")
else:
    print(f"Single GPU         : {num_gpus} GPU")

try:
    model = torch.compile(model, mode="default")
    print("torch.compile: enabled")
except Exception as e:
    print(f"torch.compile: skipped ({e})")

# --- optimizer + scheduler ---
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs),
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs - warmup_epochs, eta_min=1e-6),
    ],
    milestones=[warmup_epochs],
)

# --- loss: FocalDice + OHEM ---
criterion = FocalDiceLoss(
    num_classes=NUM_CLASSES,
    dice_weight=dice_w,
    gamma=focal_gamma,
    class_weights=class_weights.to(device),
    label_smoothing=label_smoothing,
    ohem_ratio=ohem_ratio,
)

scaler = torch.amp.GradScaler("cuda", enabled=(not use_bf16 and device.type == "cuda"))

steps_per_epoch = len(train_loader)
print(f"\nModel     : UNetV2(base={base_width})  |  Params: {num_params:,}")
print(f"Device    : {device}  |  AMP: {amp_dtype}  |  cudnn.benchmark: ON")
print(f"Training  : {epochs} epochs × {steps_per_epoch} steps  (batch={batch_size})")
print(f"Warmup    : {warmup_epochs} epochs = {warmup_epochs * steps_per_epoch} gradient steps")
print(f"Loss      : FocalDice(gamma={focal_gamma}, dice_w={dice_w}, OHEM={ohem_ratio})")
print(f"Extras    : EMA(decay={ema_decay})  TTA(hflip+label_swap)  DeepSupervision")

wandb.init(
    project="face-parsing-unet",
    name=f"unetv2_b{batch_size}_lr{lr}_warmup{warmup_epochs}_hardsampler_{num_gpus}gpu",
    config={
        "model":              "UNetV2",
        "num_classes":        NUM_CLASSES,
        "base_width":         base_width,
        "dropout":            dropout,
        "params":             num_params,
        "loss":               "FocalDice+OHEM",
        "focal_gamma":        focal_gamma,
        "dice_weight":        dice_w,
        "ohem_ratio":         ohem_ratio,
        "aux_weight":         aux_weight,
        "class_weights":      "sqrt(median/freq)",
        "hard_classes":       sorted(HARD_CLASSES),
        "hard_class_weight":  hard_class_weight,
        "lr":                 lr,
        "warmup_epochs":      warmup_epochs,
        "warmup_steps":       warmup_epochs * steps_per_epoch,
        "scheduler":          "LinearWarmup+CosineAnnealingLR",
        "batch_size":         batch_size,
        "epochs":             epochs,
        "steps_per_epoch":    steps_per_epoch,
        "ema_decay":          ema_decay,
        "tta":                "hflip+label_swap",
        "deep_supervision":   True,
        "seed":               seed,
        "amp_dtype":          str(amp_dtype),
        "augmentation":       "hflip+label_swap, ShiftScaleRotate, ColorJitter, GaussianBlur",
        "num_gpus":           num_gpus,
    },
)

In [ ]:
from contextlib import contextmanager
import numpy as np

# ---------------------------------------------------------------------------
# EMA (Exponential Moving Average) weight tracker
# ---------------------------------------------------------------------------

class EMAKeeper:
    def __init__(self, model, decay: float = 0.999):
        self.decay = decay
        self.num_updates = 0
        self.shadow = {
            name: param.data.clone()
            for name, param in model.named_parameters()
            if param.requires_grad
        }

    @torch.no_grad()
    def update(self, model):
        self.num_updates += 1
        decay = min(self.decay, (1 + self.num_updates) / (10 + self.num_updates))
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name].mul_(decay).add_(param.data, alpha=1 - decay)

    @contextmanager
    def applied(self, model):
        original = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.shadow[name])
        try:
            yield
        finally:
            for name, param in model.named_parameters():
                if param.requires_grad:
                    param.data.copy_(original[name])


# ---------------------------------------------------------------------------
# Validate with EMA weights + TTA
# ---------------------------------------------------------------------------

_FLIP_PAIRS = [(4, 5), (6, 7), (8, 9)]


@torch.no_grad()
def validate(model, loader, device, ema: EMAKeeper | None = None):
    ctx = ema.applied(model) if ema is not None else contextmanager(lambda: (yield))()
    model.eval()
    cm_total = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64)

    with ctx:
        for imgs, masks, _ in loader:
            imgs  = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"):
                logits = model(imgs)
                logits_flip = model(imgs.flip(-1)).flip(-1)
                for a, b in _FLIP_PAIRS:
                    logits_flip[:, [a, b]] = logits_flip[:, [b, a]]
                logits = (logits + logits_flip) * 0.5

            pred = logits.argmax(dim=1)
            cm_total += confusion_matrix(pred.cpu(), masks.cpu(), num_classes=NUM_CLASSES)

    macro_f1, per_class_f1 = f1_macro_from_cm(cm_total)
    val_fscore, _, _ = fbeta_present_gt_from_cm(cm_total, beta=1.0)
    return macro_f1, per_class_f1, val_fscore


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------

def _downsample_mask(masks: torch.Tensor, size: tuple) -> torch.Tensor:
    return F.interpolate(
        masks.float().unsqueeze(1), size=size, mode="nearest"
    ).squeeze(1).long()


def _model_state_dict(model):
    m = model.module if isinstance(model, nn.DataParallel) else model
    return m.state_dict()


def train():
    print(f"Trainable params: {num_params:,}")
    best_f1 = -1.0
    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
    ema = EMAKeeper(model, decay=ema_decay)

    for epoch in range(1, epochs + 1):
        model.train()
        t0      = time.time()
        running = 0.0

        for imgs, masks, _ in train_loader:
            imgs  = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"):
                outputs = model(imgs)
                if isinstance(outputs, tuple):
                    main, aux1, aux2 = outputs
                    masks_h8 = _downsample_mask(masks, aux1.shape[-2:])
                    masks_h4 = _downsample_mask(masks, aux2.shape[-2:])
                    loss = (criterion(main, masks)
                            + aux_weight * criterion(aux1, masks_h8)
                            + aux_weight * criterion(aux2, masks_h4))
                else:
                    loss = criterion(outputs, masks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            ema.update(model)
            running += loss.item()

        train_loss = running / max(1, len(train_loader))
        scheduler.step()
        val_f1, per_class_f1, val_fscore = validate(model, val_loader, device, ema=ema)
        elapsed = time.time() - t0

        wandb.log({
            "epoch":          epoch,
            "train/loss":     train_loss,
            "val/macro_f1":   val_f1,
            "val/fscore":     val_fscore,
            "time/epoch_sec": elapsed,
            "lr":             scheduler.get_last_lr()[0],
            "ema/decay":      min(ema_decay, (1 + ema.num_updates) / (10 + ema.num_updates)),
        })

        print(f"[{epoch:03d}/{epochs}] loss={train_loss:.4f}  "
              f"val_F1={val_f1:.4f}  fscore={val_fscore:.4f}  "
              f"lr={scheduler.get_last_lr()[0]:.2e}  time={elapsed:.1f}s")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                "model":          _model_state_dict(model),
                "ema_shadow":     ema.shadow,
                "optimizer":      optimizer.state_dict(),
                "epoch":          epoch,
                "best_f1":        best_f1,
                "per_class_f1":   per_class_f1.tolist(),
                "config":         dict(wandb.config),
            }, ckpt_path)
            print(f"  saved {ckpt_path}  (F1={best_f1:.4f})")

    print(f"\nBest val macro-F1: {best_f1:.4f}")
    wandb.finish()


In [ ]:
train()

Trainable params: 1,796,664


W0316 20:59:29.873000 294165 /scratch-share/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/_logging/_internal.py:1081] [0/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored
/home/msai/qiao0042/QIAO0042/.conda/envs/face_parsing/lib/python3.10/site-packages/torch/_dynamo/variables/functions.py:725: UserWarning: Graph break due to unsupported builtin sys._getframe. This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind). If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround. If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use torch.compiler.allow_in_graph.
  torch._dynamo.utils.warn_once(msg)


[001/300] loss=2.5545  val_F1=0.0000  lr=5.80e-03  time=33.7s
  saved checkpoints/v2_best.pt  (F1=0.0000)
[002/300] loss=2.4901  val_F1=0.0030  lr=7.60e-03  time=27.0s
  saved checkpoints/v2_best.pt  (F1=0.0030)
[003/300] loss=2.5107  val_F1=0.0115  lr=9.40e-03  time=27.0s
  saved checkpoints/v2_best.pt  (F1=0.0115)
[004/300] loss=2.6828  val_F1=0.0076  lr=1.12e-02  time=26.8s
[005/300] loss=2.9839  val_F1=0.0026  lr=1.30e-02  time=26.4s
[006/300] loss=3.4928  val_F1=0.0023  lr=1.48e-02  time=27.4s
[007/300] loss=4.0221  val_F1=0.0020  lr=1.66e-02  time=26.6s
